# ITCHI v0.1 — Cristóbal 2020 (NA) y Odile 2014 (EP): MSWEP V3.16 a 6 h, rampas R3/R4 y radios ROCLOUD/W06

Cambios respecto a `02_cristobal_odile_rocloud_vs_w06.ipynb`:

1. Precipitación: acumulados de 6 h de MSWEP V3.16 ya calculados (`D:/MSWEP_V316_6H/mswep_v3_6h_YYYY.nc`, los mismos del análisis de declaratorias), tomados en cada tiempo sinóptico (snapshots), en lugar de MSWEP V2.8 diario. No se leen archivos horarios.
2. Climatología: percentiles de 6 h de MSWEP V3.16 (mayo–noviembre, 2000–2024), no los diarios de V2.8.
3. Dos rampas de peligro de precipitación $H_P$, evaluadas con los mismos percentiles:
   - `R3` (vigente): Q90 → 0, Q95 → 0.5, Q99 → 1.
   - `R4` (propuesta): Q75, Q90, Q95, Q99. Los valores de $H_P$ en cada nodo son un parámetro (sección 0).
4. Solo dos hipótesis de radio de atribución:
   - `w06`: radio exterior W06 simétrico.
   - `rocloud_w06floor`: ROCLOUD por cuadrante; si $R_{\mathrm{ROCLOUD},q} < R_{\mathrm{W06}}$, o el cuadrante no tiene detección, se usa $R_{\mathrm{W06}}$. Es decir, $R_q = \max(R_{\mathrm{ROCLOUD},q},\, R_{\mathrm{W06}})$.
5. El peligro de viento usa solo el perfil W06 (se elimina Chavas).

Diseño factorial: 2 radios × 2 rampas = 4 experimentos por caso.

Consecuencias del diseño que conviene tener presentes al leer los resultados:

- Como $R_q \geq R_{\mathrm{W06}}$, la región de atribución de `rocloud_w06floor` contiene a la de `w06`. Con $\mathrm{ITCHI} = 1-(1-H_{dir})(1-H_{ind})$, la diferencia `rocloud_w06floor − w06` es no negativa celda a celda: solo mide el área adicional que aporta ROCLOUD más allá de W06.
- Con los nodos por defecto de `R4`, $H_P^{R4} \geq H_P^{R3}$ para cualquier $P$; la diferencia `R4 − R3` también es no negativa. Ambas propiedades se verifican numéricamente (sección 10).
- En el notebook 02, el radio W06 `calib_r34` tenía un sesgo de +120 km frente a `Rp` de ROCLOUD en Cristóbal. Si se mantiene esa configuración, el piso W06 reemplazará la mayoría de los cuadrantes y las dos hipótesis de radio convergerán. La sección 3 cuantifica la fracción de cuadrantes reemplazados.

Se ejecuta un caso por corrida: elige `CASE` en la celda de parámetros.

In [25]:
import os
import re
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from itchi.aggregation import compute_event_accumulated, compute_event_max
from itchi.config import get_project_root, load_default_config
from itchi.constants import DEFAULT_EPSILON, QUADRANTS
from itchi.geometry import EARTH_RADIUS_KM, build_geometry_fields
from itchi.ibtracs import (
    _decode_text_sequence,  # helper privado, solo para decodificar nombres
    get_ibtracs_storm_ids,
    ibtracs_to_track_dataframe,
    read_ibtracs_dataset,
)
from itchi.mswep import (
    extract_mswep_precipitation,
    infer_mswep_lat_lon_names,
    read_mswep_dataset,
    standardize_mswep_time,
    subset_mswep_domain,
)
from itchi.parametric_radii import (
    add_parametric_radii,
    compute_w06_wind_hazard,
    parametric_direct_fallback_km,
)
from itchi.pipeline import compute_itchi_snapshot_from_grid
from itchi.plotting import plot_itchi_field
from itchi.precipitation import compute_precipitation_hazard
from itchi.radii import resolve_direct_radius
from itchi.rocloud import (
    DEFAULT_ROCLOUD_COLUMN_MAP,
    ROCLOUD_TEXT_COLUMNS,
    standardize_rocloud_text_dataframe,
)
from itchi.tracks import DEFAULT_R34_COLUMN_MAP, filter_synoptic_times
from itchi.units import convert_quadrant_radii_to_km
from itchi.wind_profiles import V_R34_MS

## 0. Parámetros

In [27]:
ROOT = get_project_root()
CFG = load_default_config()
PR = CFG.get("parametric_radii", {})

CASES = {
    "CRISTOBAL2020": dict(name="CRISTOBAL", season=2020, basin="NA", atcf="AL032020",
                          t_check="2020-06-03 12:00"),   # primera entrada a tierra (Campeche)
    "ODILE2014": dict(name="ODILE", season=2014, basin="EP", atcf="EP152014",
                      t_check="2014-09-15 00:00"),       # cerca de la entrada a tierra (BCS)
}
CASE = "CRISTOBAL2020"                     # cambia a "ODILE2014" para el Pacífico
C = CASES[CASE]
T_CHECK = pd.Timestamp(C["t_check"])

# --- Trayectoria y ROCLOUD -------------------------------------------------
IBTRACS_FILE = ROOT / CFG["paths"]["tracks"] / f"IBTrACS.{C['basin']}.v04r01.nc"
ROCLOUD_FILE = ROOT / "data" / "external" / "Rocloud" / f"ROCLOUD_{C['basin']}.csv"
ROCLOUD_MWS_TO_KT = 1.0 / 1.852           # MWS de ROCLOUD está en km/h (46.25 = 25 kt)

# --- Radios W06 ("el radio W06 que definimos") -----------------------------
W06_CONFIGS = {
    "pa21": dict(use_observed_rmw=False, calibrate_to_r34=False, rmax_variant="pa21"),
    "obs_rmw": dict(use_observed_rmw=True, calibrate_to_r34=False),
    "calib_r34": dict(use_observed_rmw=True, calibrate_to_r34=True),
}
W06_CHOICE = "calib_r34"                   # misma configuración que el ITCHI del notebook 02
PREFIX = f"{W06_CHOICE}_"
THRESHOLDS = {"r_outer_km": PR.get("outer_threshold_ms", 2.0), "r34_km": V_R34_MS}
RMW_UNIT = PR.get("rmw_unit", "km")
assert RMW_UNIT == "km", "rmw_km ya sale en km de ibtracs.py"

# --- MSWEP V3.16 a 6 h (mismos archivos que en declaratorias) ------------
# mswep_v3_6h_YYYY.nc (scripts/mswep_v3_6h_accum.py): variable "precipitation" (mm),
# time = fin de la ventana (valid time), ventana (T-6h, T], solo mayo-noviembre.
def local_path(path) -> Path:
    # "D:/..." -> "/mnt/d/..." cuando se ejecuta en WSL/Linux; sin cambios en Windows
    s = str(path).replace("\\", "/")
    if os.name != "nt" and re.match(r"^[A-Za-z]:/", s):
        return Path(f"/mnt/{s[0].lower()}") / s[3:]
    return Path(s)


MSWEP_6H_DIR = Path("/mnt/d/MSWEP_V316_6H")
assert MSWEP_6H_DIR.is_dir(), f"no existe {MSWEP_6H_DIR}"
MSWEP_6H_PATTERN = "mswep_v3_6h_{year}.nc"
PRECIP_VAR = "precipitation"

# --- Percentiles de 6 h de MSWEP V3.16 ------------------------------------
CLIM_FILE = ROOT / "data" / "processed" / "mswep_v3_6h_percentiles_may_nov.nc"
CLIM_VAR = "p_wet"                         # periodos húmedos; NO usar percentiles sobre todos los periodos
CLIM_VAR_NAMES = {}                        # solo si hay una variable por percentil: {"Q75": "p75", ...}
CLIM_DOMAIN = dict(lon_min=-140.0, lon_max=-60.0, lat_min=5.0, lat_max=35.0)
PAD_DEG = 10.0
DROP_SNAPSHOTS_OUTSIDE_DOMAIN = True       # centros fuera del dominio de los percentiles

# --- Rampas de H_P: {percentil: H_P en el nodo}, nodos en orden creciente ---
RAMPS = {
    "R3": {"Q90": 0.0, "Q95": 0.5, "Q99": 1.0},                   # vigente
    "R4": {"Q75": 0.0, "Q90": 0.25, "Q95": 0.5, "Q99": 1.0},      # propuesta (conserva Q95=0.5, Q99=1)
    # alternativa equiespaciada: {"Q75": 0.0, "Q90": 1/3, "Q95": 2/3, "Q99": 1.0}
}
ATTRIBUTIONS = ("w06", "rocloud_w06floor")
EXPERIMENTS = [(a, r) for a in ATTRIBUTIONS for r in RAMPS]
REFERENCE = ("w06", "R3")

# --- Salidas ---------------------------------------------------------------
TAG = "v316_6h"
FIG_DIR = ROOT / CFG["outputs"]["figures"] / CASE.lower() / TAG
FIG_DIR.mkdir(parents=True, exist_ok=True)
DIAG_DIR = ROOT / CFG["outputs"]["diagnostics"]
DIAG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = ROOT / CFG["outputs"]["event_products"]
OUT_DIR.mkdir(parents=True, exist_ok=True)

OSError: [Errno 19] No such device: '/mnt/d/MSWEP_V316_6H'

## 0b. Estilo de figuras

Todas las figuras usan las convenciones de `itchi.plotting`: `configure_map_axis` con un `MapStyle` (tierra, costas, fronteras, ticks y cuadrícula), `batlow` para campos de peligro en [0, 1] y `plot_itchi_siat_levels` para niveles. Cambios respecto a la versión anterior de las figuras:

- Colorbars pequeñas en un eje insertado (`CBAR`: alto, grosor y separación como fracción del mapa).
- Ceros de los campos de peligro y lluvia < `P_MIN_PLOT` transparentes, para que se vean tierra y océano.
- Tamaño de figura derivado de la extensión del mapa (sin franjas vacías); en rejillas, etiquetas de lon/lat solo en los bordes.
- Regiones directa (negro) y de atribución (magenta) dibujadas como contornos de las máscaras del pipeline, es decir, exactamente las regiones usadas en el cálculo.
- Colores fijos por experimento: color = radio de atribución, estilo de línea = rampa.
- Mapas individuales a detalle (`plot_snapshot_map`, `plot_event_map`) que devuelven `None` con aviso en lugar de detener la ejecución.

Las funciones de esta celda usan variables definidas más abajo (`bbox`, `LON2D`, `events`, ...) solo al llamarlas.

In [ ]:
import warnings
from dataclasses import replace

import matplotlib.dates as mdates
from matplotlib import patheffects
from matplotlib.artist import Artist
from matplotlib.lines import Line2D

from itchi.plotting import (
    SIAT_DANGER_LABELS,
    MapStyle,
    configure_map_axis,
    get_crameri_colormap,
    get_default_itchi_colormap,
    plot_itchi_siat_levels,
)

PC = ccrs.PlateCarree()

# --- Mapas: estilo de itchi.plotting ----------------------------------------
MAP_STYLE = MapStyle(
    lon_tick_spacing=4.0, lat_tick_spacing=4.0, grid_spacing=4.0,
    grid_color="w", grid_alpha=0.6, label_size=9.0, tick_size=4.0,
)
CBAR = dict(height=0.70, width=0.035, pad=0.03, labelsize=9)   # fracciones del eje del mapa
MAP_WIDTH_IN = 4.2                          # ancho de cada mapa en las rejillas (pulgadas)
SNAP_HALF_WIDTH_DEG = 12.0                  # semiancho de los mapas centrados en el ciclón
SNAP_FIELDS = ("P6", "H_P", "H_ind", "ITCHI")
P_MIN_PLOT = 0.1                            # mm/6 h; por debajo se deja transparente
SIAT_SHOW_ZERO = False                      # True: nivel 0 en gris sobre todo el dominio


def light_to_dark(cmap):
    # Garantiza valores bajos claros (los ceros enmascarados no contrastan con el fondo)
    lum = lambda c: 0.2126 * c[0] + 0.7152 * c[1] + 0.0722 * c[2]  # noqa: E731
    return cmap if lum(cmap(0.0)) >= lum(cmap(1.0)) else cmap.reversed()


CMAP_HAZARD = get_default_itchi_colormap()                   # ITCHI, H_P, H_dir, H_ind
CMAP_PRECIP = light_to_dark(get_crameri_colormap("devon"))   # P6 y percentiles
CMAP_DELTA = light_to_dark(get_crameri_colormap("lajolla"))  # diferencias >= 0
CMAP_DIVERGING = get_crameri_colormap("vik")                 # diferencias con signo

# --- Colores por experimento (series de tiempo y leyendas) ------------------
ATTR_COLORS = {"w06": "#1f78b4", "rocloud_w06floor": "#d7301f"}
RAMP_LS = {"R3": "-", "R4": "--"}
RAMP_COLORS = {"R3": "0.25", "R4": "#e6550d"}
REGION_STYLE = {"direct": dict(colors="k", linewidths=1.2),
                "attribution": dict(colors="m", linewidths=1.5)}

FIELD_LABELS = {
    "P6": "P (mm/6 h)", "H_P": "$H_P$", "H_W": "$H_W$", "H_dir": "$H_{dir}$",
    "H_ind": "$H_{ind}$", "ITCHI": "ITCHI", "ITCHI_max": "ITCHI$_{max}$",
    "ITCHI_acc": "ITCHI$_{acc}$", "SIAT": "nivel ITCHI",
}
HAZARD_FIELDS = {"H_P", "H_W", "H_dir", "H_ind", "ITCHI", "ITCHI_max", "ITCHI_acc"}


def exp_label(attribution: str, ramp: str) -> str:
    return f"{attribution} · {ramp}"


def exp_style(attribution: str, ramp: str) -> dict:
    return dict(color=ATTR_COLORS.get(attribution, "0.4"), ls=RAMP_LS.get(ramp, "-"),
                label=exp_label(attribution, ramp))


def format_time_axis(ax) -> None:
    locator = mdates.AutoDateLocator(minticks=4, maxticks=9)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.grid(alpha=0.3, lw=0.6)


# --- Extensión de los mapas --------------------------------------------------
def style_for_bounds(lon_min, lon_max, lat_min, lat_max, base: MapStyle = MAP_STYLE):
    # Extensión alineada a múltiplos del espaciado de ticks
    sx, sy = base.lon_tick_spacing, base.lat_tick_spacing
    return replace(base,
                   lon_extent=(np.floor(lon_min / sx) * sx, np.ceil(lon_max / sx) * sx),
                   lat_extent=(np.floor(lat_min / sy) * sy, np.ceil(lat_max / sy) * sy))


def case_style(base: MapStyle = MAP_STYLE) -> MapStyle:
    return style_for_bounds(bbox["lon_min"], bbox["lon_max"], bbox["lat_min"],
                            bbox["lat_max"], base)


def position_style(row: pd.Series, half_width: float = SNAP_HALF_WIDTH_DEG,
                   base: MapStyle = MAP_STYLE) -> MapStyle:
    return style_for_bounds(row["lon"] - half_width, row["lon"] + half_width,
                            row["lat"] - 0.75 * half_width, row["lat"] + 0.75 * half_width,
                            base)


# --- Figuras de mapas ---------------------------------------------------------
def new_map_figure(nrows: int, ncols: int, style: MapStyle | None = None,
                   map_width: float = MAP_WIDTH_IN, wspace_in: float = 1.0,
                   hspace_in: float = 0.45, top_in: float = 1.0, bottom_in: float = 0.45,
                   left_in: float = 0.65, right_in: float = 1.0):
    # Tamaño de figura derivado de la extensión: cada eje mide exactamente lo que el mapa,
    # sin franjas vacías; márgenes y separaciones en pulgadas (espacio para colorbars)
    resolved = style or case_style()
    (x0, x1), (y0, y1) = resolved.lon_extent, resolved.lat_extent
    map_height = map_width * (y1 - y0) / (x1 - x0)
    width = left_in + ncols * map_width + (ncols - 1) * wspace_in + right_in
    height = top_in + nrows * map_height + (nrows - 1) * hspace_in + bottom_in
    fig, axes = plt.subplots(nrows, ncols, figsize=(width, height),
                             subplot_kw={"projection": PC}, squeeze=False)
    fig.subplots_adjust(left=left_in / width, right=1.0 - right_in / width,
                        bottom=bottom_in / height, top=1.0 - top_in / height,
                        wspace=wspace_in / map_width, hspace=hspace_in / map_height)
    for ax in axes.flat:
        configure_map_axis(ax, crs=PC, style=resolved)
    return fig, axes


def hide_inner_labels(axes) -> None:
    # Con extensión común, etiquetas de lon solo abajo y de lat solo a la izquierda
    for ax in axes[:-1, :].flat:
        ax.tick_params(labelbottom=False)
    for ax in axes[:, 1:].flat:
        ax.tick_params(labelleft=False)


def finish_map_figure(fig, title: str, handles=None, ncol: int = 4) -> None:
    height = fig.get_size_inches()[1]
    fig.suptitle(title, fontsize=13, y=1.0 - 0.18 / height)
    if handles:
        fig.legend(handles=handles, loc="upper center", ncol=ncol, fontsize=9,
                   frameon=False, bbox_to_anchor=(0.5, 1.0 - 0.45 / height))


def add_small_colorbar(ax, mappable, label: str, ticks=None, ticklabels=None,
                       extend: str = "neither", **opts):
    # Colorbar en un eje insertado: alto = fracción del alto del mapa, centrada
    o = {**CBAR, **opts}
    cax = ax.inset_axes([1.0 + o["pad"], (1.0 - o["height"]) / 2, o["width"], o["height"]])
    cbar = ax.figure.colorbar(mappable, cax=cax, ticks=ticks, extend=extend)
    cbar.set_label(label, fontsize=o["labelsize"])
    cbar.ax.tick_params(labelsize=o["labelsize"] - 1)
    if ticklabels is not None:
        cbar.set_ticklabels(ticklabels)
    return cbar


def add_halo(artist, linewidth: float = 2.6) -> None:
    # Borde blanco para que líneas y contornos se lean sobre cualquier fondo
    effects = [patheffects.withStroke(linewidth=linewidth, foreground="w")]
    if isinstance(artist, Artist):                  # ContourSet es Artist en matplotlib >= 3.8
        artist.set_path_effects(effects)
    else:
        for collection in artist.collections:
            collection.set_path_effects(effects)


def draw_track(ax, current: pd.Series | None = None) -> None:
    (line,) = ax.plot(track["lon"], track["lat"], "-", color="0.15", lw=0.9, marker="o",
                      ms=1.8, transform=PC, zorder=5)
    add_halo(line, 2.2)
    if current is not None:
        ax.plot(current["lon"], current["lat"], marker="*", color="w", mec="k", mew=0.8,
                ms=13, transform=PC, zorder=7)


def draw_regions(ax, m_direct, m_region, linestyle: str = "-") -> None:
    # Contornos exactos de las máscaras del pipeline (cuadrantes por lat/lon relativas)
    for mask, key in ((m_region, "attribution"), (m_direct, "direct")):
        mask = np.asarray(mask, dtype=float)
        if mask.max() > 0:
            contours = ax.contour(LON2D, LAT2D, mask, levels=[0.5], linestyles=linestyle,
                                  transform=PC, zorder=6, **REGION_STYLE[key])
            add_halo(contours, 3.0)


def map_legend_handles(regions: bool = True, position: bool = True,
                       second_attribution: str | None = None):
    handles = [Line2D([], [], color="0.15", lw=0.9, marker="o", ms=2, label="trayectoria")]
    if position:
        handles.append(Line2D([], [], ls="none", marker="*", mfc="w", mec="k", ms=12,
                              label="posición"))
    if regions:
        handles += [Line2D([], [], color="k", lw=1.2, label="región directa"),
                    Line2D([], [], color="m", lw=1.5, label="región de atribución")]
    if second_attribution:
        handles.append(Line2D([], [], color="m", lw=1.5, ls="--",
                              label=f"atribución {second_attribution}"))
    return handles


def draw_field(ax, field: str, values, vmax: float | None = None, cbar: bool = True,
               cbar_opts: dict | None = None):
    opts = cbar_opts or {}
    values = np.asarray(values, dtype=float)
    if field == "P6":
        finite = values[np.isfinite(values)]
        vmax = vmax or max(1.0, float(np.percentile(finite, 99.5)) if finite.size else 1.0)
        mesh = ax.pcolormesh(LON2D, LAT2D, np.where(values >= P_MIN_PLOT, values, np.nan),
                             cmap=CMAP_PRECIP, vmin=0, vmax=vmax, shading="auto",
                             transform=PC, zorder=1)
        if cbar:
            add_small_colorbar(ax, mesh, FIELD_LABELS["P6"], extend="max", **opts)
    elif field == "SIAT":
        shown = values if SIAT_SHOW_ZERO else np.where(values > 0, values, np.nan)
        mesh = plot_itchi_siat_levels(shown, ax, lon=LON2D, lat=LAT2D, crs=PC,
                                      add_colorbar=False, zorder=1)
        if cbar:
            add_small_colorbar(ax, mesh, FIELD_LABELS["SIAT"], ticks=np.arange(0.5, 6.5, 1.0),
                               ticklabels=[lab.split(" ", 1)[0] for lab in SIAT_DANGER_LABELS],
                               **opts)
    elif field in HAZARD_FIELDS:
        mesh = plot_itchi_field(np.where(values > 0, values, np.nan), ax, lon=LON2D, lat=LAT2D,
                                crs=PC, cmap=CMAP_HAZARD, vmin=0.0, vmax=1.0,
                                add_colorbar=False, zorder=1)
        if cbar:
            add_small_colorbar(ax, mesh, FIELD_LABELS.get(field, field), **opts)
    else:
        raise ValueError(f"campo no soportado: {field}")
    return mesh


def draw_delta(ax, values, vmax: float, label: str = "ΔITCHI$_{max}$", cbar_opts=None):
    # Diferencias >= 0 en escala secuencial (ceros transparentes); con signo, divergente
    values = np.asarray(values, dtype=float)
    vmax = float(np.ceil(max(vmax, 0.05) * 20) / 20)
    if np.nanmin(values) < -1e-6:
        mesh = ax.pcolormesh(LON2D, LAT2D, np.where(np.abs(values) > 1e-6, values, np.nan),
                             cmap=CMAP_DIVERGING, vmin=-vmax, vmax=vmax, shading="auto",
                             transform=PC, zorder=1)
    else:
        mesh = ax.pcolormesh(LON2D, LAT2D, np.where(values > 1e-6, values, np.nan),
                             cmap=CMAP_DELTA, vmin=0.0, vmax=vmax, shading="auto",
                             transform=PC, zorder=1)
    add_small_colorbar(ax, mesh, label, **(cbar_opts or {}))
    return mesh


# --- Acceso a los productos por experimento y tiempo -----------------------------
def time_index(ev, t) -> int | None:
    idx = np.flatnonzero(ev["time"].values == np.datetime64(pd.Timestamp(t)))
    return int(idx[0]) if idx.size else None


def snapshot_fields(exp, t) -> dict | None:
    ev = events.get(exp)
    i = None if ev is None else time_index(ev, t)
    if i is None:
        return None
    out = {k: np.asarray(ev[k].values[i]) for k in FIELDS_F + FIELDS_B}
    out["P6"] = PRECIP6[pd.Timestamp(t)]
    out["SIAT"] = out["ITCHI"]
    out["region"] = out["M_direct"] | out["M_indirect"]
    return out


def track_row(t) -> pd.Series | None:
    rows = track[track["time"] == pd.Timestamp(t)]
    return None if rows.empty else rows.iloc[0]


# --- Figuras compuestas ------------------------------------------------------------
def plot_event_grid(field: str = "ITCHI_max", siat: bool = False):
    # Rejilla radios x rampas de un producto de evento
    fig, axes = new_map_figure(len(ATTRIBUTIONS), len(RAMPS))
    for i, a in enumerate(ATTRIBUTIONS):
        for j, r in enumerate(RAMPS):
            ax = axes[i, j]
            draw_field(ax, "SIAT" if siat else field, events[(a, r)][field].values)
            draw_track(ax)
            ax.set_title(exp_label(a, r), fontsize=10.5)
    hide_inner_labels(axes)
    kind = "niveles SIAT" if siat else FIELD_LABELS.get(field, field)
    finish_map_figure(fig, f"{C['name']} {C['season']} — {kind}, MSWEP V3.16 6 h",
                      handles=map_legend_handles(regions=False, position=False))
    return fig


def plot_snapshot_grid(t, fields=SNAP_FIELDS, experiments=None,
                       half_width: float = SNAP_HALF_WIDTH_DEG):
    # Rejilla experimentos x campos para una posición; None si no hay posición
    experiments = experiments or EXPERIMENTS
    row = track_row(t)
    if row is None:
        warnings.warn(f"sin posición en {t}; se omite la figura")
        return None
    p_finite = PRECIP6[pd.Timestamp(t)][np.isfinite(PRECIP6[pd.Timestamp(t)])]
    p_vmax = max(1.0, float(np.percentile(p_finite, 99.5))) if p_finite.size else 1.0
    fig, axes = new_map_figure(len(experiments), len(fields),
                               style=position_style(row, half_width))
    for i, exp in enumerate(experiments):
        snap = snapshot_fields(exp, t)
        for j, field in enumerate(fields):
            ax = axes[i, j]
            if snap is None:
                ax.set_title(f"{exp_label(*exp)}: sin snapshot", fontsize=9)
                continue
            draw_field(ax, field, snap[field], vmax=p_vmax if field == "P6" else None)
            if field == "P6":
                ax.contour(LON2D, LAT2D, PRECIP6[pd.Timestamp(t)] - Q["Q95"], levels=[0.0],
                           colors="#e31a1c", linewidths=0.7, transform=PC, zorder=5)
            draw_regions(ax, snap["M_direct"], snap["region"])
            draw_track(ax, current=row)
            ax.set_title(f"{FIELD_LABELS[field]} · {exp_label(*exp)}", fontsize=10)
    hide_inner_labels(axes)
    handles = map_legend_handles() + [Line2D([], [], color="#e31a1c", lw=0.7,
                                             label="P6 = Q95 (6 h)")]
    finish_map_figure(fig, f"{C['name']} {pd.Timestamp(t):%Y-%m-%d %H} UTC | "
                           f"Vmax {row['vmax_kt']:.0f} kt | MSWEP V3.16 6 h",
                      handles=handles, ncol=5)
    return fig


def plot_snapshot_map(t, exp, field: str, half_width: float = SNAP_HALF_WIDTH_DEG,
                      figsize=(9, 7), compare_attribution: bool = False):
    # Un mapa a detalle: una posición, un experimento, un campo; None si no aplica
    row, snap = track_row(t), snapshot_fields(exp, t)
    if row is None or snap is None:
        warnings.warn(f"{exp_label(*exp)}: sin snapshot en {t}; se omite el mapa")
        return None
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": PC})
    big = replace(MAP_STYLE, label_size=11.0, tick_size=6.0)
    configure_map_axis(ax, crs=PC, style=position_style(row, half_width, big))
    draw_field(ax, field, snap[field], cbar_opts={"labelsize": 11})
    if field == "P6":
        ax.contour(LON2D, LAT2D, snap["P6"] - Q["Q95"], levels=[0.0], colors="#e31a1c",
                   linewidths=0.8, transform=PC, zorder=5)
    draw_regions(ax, snap["M_direct"], snap["region"])
    other = None
    if compare_attribution:
        other = next((a for a in ATTRIBUTIONS if a != exp[0]), None)
        other_snap = snapshot_fields((other, exp[1]), t) if other else None
        if other_snap is not None:
            add_halo(ax.contour(LON2D, LAT2D, other_snap["region"].astype(float), levels=[0.5],
                                colors="m", linewidths=1.5, linestyles="--", transform=PC,
                                zorder=6), 3.0)
    draw_track(ax, current=row)
    ax.legend(handles=map_legend_handles(second_attribution=other), loc="lower left",
              fontsize=9, framealpha=0.9)
    ax.set_title(f"{FIELD_LABELS[field]} — {exp_label(*exp)}\n{C['name']} "
                 f"{pd.Timestamp(t):%Y-%m-%d %H} UTC | Vmax {row['vmax_kt']:.0f} kt",
                 fontsize=12)
    return fig


def plot_event_map(exp, field: str = "ITCHI_max", siat: bool = False, figsize=(9, 7.5)):
    # Un producto de evento a detalle; None si el experimento no existe
    ev = events.get(exp)
    if ev is None:
        warnings.warn(f"{exp}: experimento no calculado; se omite el mapa")
        return None
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": PC})
    configure_map_axis(ax, crs=PC, style=case_style(replace(MAP_STYLE, label_size=11.0)))
    draw_field(ax, "SIAT" if siat else field, ev[field].values, cbar_opts={"labelsize": 11})
    draw_track(ax)
    ax.legend(handles=map_legend_handles(regions=False, position=False), loc="lower left",
              fontsize=9, framealpha=0.9)
    kind = "niveles SIAT" if siat else FIELD_LABELS.get(field, field)
    ax.set_title(f"{kind} — {exp_label(*exp)}\n{C['name']} {C['season']}, MSWEP V3.16 6 h",
                 fontsize=12)
    return fig

## 1. Trayectoria IBTrACS

In [ ]:
ds_ib = read_ibtracs_dataset(IBTRACS_FILE)
sids = np.asarray(get_ibtracs_storm_ids(ds_ib))
names = np.asarray(_decode_text_sequence(ds_ib["name"].values))
seasons = np.asarray(ds_ib["season"].values).astype(int)
match = sids[(names == C["name"]) & (seasons == C["season"])]
assert match.size == 1, f"coincidencias: {match}"
SID = str(match[0])

track = ibtracs_to_track_dataframe(ds_ib, SID)          # rmw_km en km
track = filter_synoptic_times(track).dropna(subset=["vmax_kt", "lat", "lon"])

inside = track["lon"].between(CLIM_DOMAIN["lon_min"], CLIM_DOMAIN["lon_max"]) & \
         track["lat"].between(CLIM_DOMAIN["lat_min"], CLIM_DOMAIN["lat_max"])
print(f"{SID} {C['atcf']}: {len(track)} registros sinópticos; "
      f"{int((~inside).sum())} con centro fuera del dominio de los percentiles")
if DROP_SNAPSHOTS_OUTSIDE_DOMAIN:
    track = track[inside]
track = track.reset_index(drop=True)
print(track["time"].min(), "->", track["time"].max(), f"({len(track)} snapshots)")

2020154N19269 AL032020: 43 registros sinópticos; 13 con centro fuera del dominio de los percentiles
2020-06-01 18:00:00 -> 2020-06-09 00:00:00 (30 snapshots)


## 2. ROCLOUD

Mismo lector que en el notebook 02: `0.0` en un cuadrante = sin detección (se convierte a NaN), `MWS` en km h⁻¹ (se corrige a kt), `-9999` = faltante.

In [ ]:
ROC_COLS = [DEFAULT_ROCLOUD_COLUMN_MAP[q] for q in QUADRANTS]


def read_rocloud_csv(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, header=None, names=list(ROCLOUD_TEXT_COLUMNS), na_values=[-9999])
    df = standardize_rocloud_text_dataframe(raw)
    df[ROC_COLS] = df[ROC_COLS].where(df[ROC_COLS] > 0)          # 0 = sin detección
    df["vmax_kt_rocloud"] = df.pop("vmax_kt") * ROCLOUD_MWS_TO_KT  # km/h -> kt
    return df


roc_all = read_rocloud_csv(ROCLOUD_FILE)
roc = roc_all[roc_all["storm_id"] == C["atcf"]].set_index("time").sort_index()
print(f"ROCLOUD {C['basin']}: {len(roc_all)} registros; caso {C['atcf']}: {len(roc)}")
print("cuadrantes sin detección en el caso:", roc[ROC_COLS].isna().sum().to_dict())

ROCLOUD NA: 5255 registros; caso AL032020: 30
cuadrantes sin detección en el caso: {'rocloud_rne': 0, 'rocloud_rse': 1, 'rocloud_rsw': 3, 'rocloud_rnw': 1}


## 3. Radio W06 y regla de atribución por cuadrante

- `w06`: $R_q = R_{\mathrm{W06}}$ (radio exterior, umbral `THRESHOLDS["r_outer_km"]`) en los cuatro cuadrantes.
- `rocloud_w06floor`: $R_q = R_{\mathrm{ROCLOUD},q}$ si existe y $R_{\mathrm{ROCLOUD},q} \geq R_{\mathrm{W06}}$; en otro caso $R_q = R_{\mathrm{W06}}$. Los tiempos sin registro ROCLOUD quedan, por tanto, iguales a `w06`; se registran para no confundirlos con reemplazos por la regla del piso.

En ambos casos se exige $R_q \geq R_{direct,q}$ (control de calidad del pipeline); los cuadrantes recortados por esta condición se cuentan.

In [ ]:
track = add_parametric_radii(track, rmw_unit=RMW_UNIT, r34_unit="nm", thresholds_ms=THRESHOLDS,
                             prefix=PREFIX, **W06_CONFIGS[W06_CHOICE])
track = track.merge(roc[["rocloud_mean_km"] + ROC_COLS].reset_index(), on="time", how="left")


def rmw_km(row: pd.Series) -> float:
    return float(row["rmw_km"]) if np.isfinite(row["rmw_km"]) else float(row[f"{PREFIX}rmax_km"])


def direct_radii_km(row: pd.Series) -> dict:
    r34_km = convert_quadrant_radii_to_km(
        {q: row[DEFAULT_R34_COLUMN_MAP[q]] for q in QUADRANTS}, input_unit="nm")
    return resolve_direct_radius(
        r34_km, vmax_kt=float(row["vmax_kt"]), rmw_km=rmw_km(row),
        fallback_radius_km=parametric_direct_fallback_km(row, f"{PREFIX}r34_km"),
    ).radii


def attribution_radii_km(row: pd.Series, mode: str):
    # Devuelve (radios por cuadrante en km, estado por cuadrante, cuadrantes recortados a R_direct)
    r_w06 = float(row[f"{PREFIX}r_outer_km"])
    if not np.isfinite(r_w06):
        return None, {q: "no_w06" for q in QUADRANTS}, []
    if mode == "w06":
        radii = {q: r_w06 for q in QUADRANTS}
        status = {q: "w06" for q in QUADRANTS}
    elif mode == "rocloud_w06floor":
        radii, status = {}, {}
        for q in QUADRANTS:
            r_roc = float(row[DEFAULT_ROCLOUD_COLUMN_MAP[q]])
            if not np.isfinite(r_roc):
                radii[q], status[q] = r_w06, "sin_rocloud"
            elif r_roc < r_w06:
                radii[q], status[q] = r_w06, "piso_w06"
            else:
                radii[q], status[q] = r_roc, "rocloud"
    else:
        raise ValueError(mode)
    direct = direct_radii_km(row)
    clipped = [q for q in QUADRANTS if radii[q] < direct[q]]
    return {q: max(radii[q], direct[q]) for q in QUADRANTS}, status, clipped


# Radios de atribución por snapshot e hipótesis
RADII, rad_rows = {}, []
for _, row in track.iterrows():
    for mode in ATTRIBUTIONS:
        radii, status, clipped = attribution_radii_km(row, mode)
        RADII[(row["time"], mode)] = radii
        for q in QUADRANTS:
            rad_rows.append(dict(time=row["time"], mode=mode, quadrant=q,
                                 r_w06_km=row[f"{PREFIX}r_outer_km"],
                                 r_rocloud_km=row[DEFAULT_ROCLOUD_COLUMN_MAP[q]],
                                 r_final_km=np.nan if radii is None else radii[q],
                                 status=status[q], clipped_to_direct=q in clipped))
radii_log = pd.DataFrame(rad_rows)
radii_log.to_csv(DIAG_DIR / f"attribution_radii_{CASE.lower()}_{TAG}.csv", index=False)

floor = radii_log[radii_log["mode"] == "rocloud_w06floor"]
print("estado de cuadrantes en rocloud_w06floor:")
print(floor["status"].value_counts().to_string())
has_roc = floor["status"] != "sin_rocloud"
print(f"fracción de cuadrantes con ROCLOUD reemplazados por W06: "
      f"{(floor.loc[has_roc, 'status'] == 'piso_w06').mean():.1%}")
print("cuadrantes recortados a R_direct:",
      radii_log.groupby("mode")["clipped_to_direct"].sum().to_dict())

estado de cuadrantes en rocloud_w06floor:
status
piso_w06       66
rocloud        49
sin_rocloud     5
fracción de cuadrantes con ROCLOUD reemplazados por W06: 57.4%
cuadrantes recortados a R_direct: {'rocloud_w06floor': 0, 'w06': 0}


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6.8), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.22, wspace=0.06, top=0.84)
for ax, q in zip(axes.ravel(), QUADRANTS):
    d = floor[floor["quadrant"] == q]
    rep = d["status"] == "piso_w06"
    ax.plot(d["time"], d["r_w06_km"], color=ATTR_COLORS["w06"], lw=2.0,
            label=f"W06 R$_{{out}}$ ({W06_CHOICE})")
    ax.plot(d["time"], d["r_final_km"], color=ATTR_COLORS["rocloud_w06floor"], lw=1.2,
            label="rocloud_w06floor (final)")
    ax.plot(d["time"], d["r_rocloud_km"], "o", color="0.15", ms=3.5, label="ROCLOUD")
    ax.plot(d.loc[rep, "time"], d.loc[rep, "r_rocloud_km"], "x",
            color=ATTR_COLORS["rocloud_w06floor"], ms=7, mew=1.6,
            label="ROCLOUD < W06 (reemplazado)")
    ax.text(0.015, 0.95, q, transform=ax.transAxes, va="top", fontsize=11, fontweight="bold")
    format_time_axis(ax)
for ax in axes[:, 0]:
    ax.set_ylabel("radio (km)")
legend = {}
for ax in axes.flat:
    for handle, label in zip(*ax.get_legend_handles_labels()):
        legend.setdefault(label, handle)
fig.legend(legend.values(), legend.keys(), loc="upper center", ncol=4, fontsize=9,
           frameon=False, bbox_to_anchor=(0.5, 0.935))
fig.suptitle(f"{C['name']} {C['season']}: radios de atribución por cuadrante", y=0.985)
fig.savefig(FIG_DIR / "attribution_radii_by_quadrant.png", dpi=150, bbox_inches="tight")

## 4. MSWEP V3.16: acumulados de 6 h por snapshot

Se leen los acumulados de 6 h ya calculados con `scripts/mswep_v3_6h_accum.py`, los mismos usados para los percentiles y para las declaratorias. En esos archivos `time` es el fin de la ventana y la ventana es $(T-6\,\mathrm{h},\,T]$, de modo que el snapshot del tiempo sinóptico $t$ es simplemente `sel(time=t)`: precipitación de las 6 h previas a la posición IBTrACS. Se exige coincidencia exacta del tiempo (sin `nearest`) y se reportan las celdas con NaN (ventanas con horas faltantes).

In [ ]:
bbox = dict(
    lon_min=max(float(track["lon"].min()) - PAD_DEG, CLIM_DOMAIN["lon_min"]),
    lon_max=min(float(track["lon"].max()) + PAD_DEG, CLIM_DOMAIN["lon_max"]),
    lat_min=max(float(track["lat"].min()) - PAD_DEG, CLIM_DOMAIN["lat_min"]),
    lat_max=min(float(track["lat"].max()) + PAD_DEG, CLIM_DOMAIN["lat_max"]),
)


def read_6h_snapshots(times: pd.Series) -> xr.DataArray:
    # Acumulados de 6 h en los tiempos sinópticos, recortados a la caja del caso
    parts = []
    times = pd.Series(pd.to_datetime(times))
    for year, t_year in times.groupby(times.dt.year):
        path = MSWEP_6H_DIR / MSWEP_6H_PATTERN.format(year=year)
        with xr.open_dataset(path) as ds:
            print(path.name, "|", ds[PRECIP_VAR].attrs.get("accumulation_window"),
                  "|", ds["time"].attrs.get("long_name"))
            da = subset_mswep_domain(ds[PRECIP_VAR], **bbox)
            missing_t = t_year[~t_year.isin(pd.DatetimeIndex(da["time"].values))]
            assert missing_t.empty, f"tiempos sin acumulado de 6 h: {list(missing_t)}"
            parts.append(da.sel(time=t_year.values).load())
    p6 = xr.concat(parts, dim="time")
    p6.attrs.update(units="mm (6 h)-1", window="(T-6h, T], T = valid_time")
    return p6


P6 = read_6h_snapshots(track["time"])
LAT_NAME, LON_NAME = infer_mswep_lat_lon_names(P6)
P0 = P6.isel(time=0, drop=True)
LON2D, LAT2D = np.meshgrid(P0[LON_NAME].values, P0[LAT_NAME].values)
PRECIP6 = {pd.Timestamp(t): np.asarray(P6.sel(time=t).values, dtype=float) for t in P6["time"].values}

nan_frac = {t: float(np.isnan(v).mean()) for t, v in PRECIP6.items()}
print(f"{len(PRECIP6)} snapshots, malla {P0.shape}; "
      f"snapshots con NaN: {sum(f > 0 for f in nan_frac.values())} (máx. {max(nan_frac.values()):.1%})")

# Área de celda (km²) para métricas en unidades físicas
dlat = np.deg2rad(abs(float(np.diff(P0[LAT_NAME].values[:2])[0])))
dlon = np.deg2rad(abs(float(np.diff(P0[LON_NAME].values[:2])[0])))
CELL_AREA_KM2 = EARTH_RADIUS_KM**2 * dlat * dlon * np.cos(np.deg2rad(LAT2D))

mswep_v3_6h_2020.nc | (T-6h, T], T = valid_time | end of 6-h accumulation window (valid time)
30 snapshots, malla (274, 228); snapshots con NaN: 0 (máx. 0.0%)


## 5. Percentiles de 6 h (MSWEP V3.16)

Se exigen todos los percentiles que usan las rampas (`Q75`, `Q90`, `Q95`, `Q99`). Si el archivo solo trae P70/P90/P95/P99/P99.9, la celda se detiene: no se sustituye Q75 por Q70 ni se interpola, porque eso cambiaría la rampa que se quiere evaluar.

Los percentiles deben ser condicionales a periodos de 6 h húmedos. Sobre todos los periodos, Q75 (y en zonas secas Q90) es prácticamente 0 mm, y la rampa `R4` se activaría con cualquier llovizna.

In [ ]:
clim_ds = xr.open_dataset(CLIM_FILE)
print(clim_ds)

REQUIRED_Q = sorted({k for ramp in RAMPS.values() for k in ramp}, key=lambda k: float(k[1:]))


def level_name(level: float, scale: float) -> str:
    return f"Q{level * scale:g}"


# Formato A: variable con dimensión de cuantil (xarray.quantile)
qdim = next((d for d in ("quantile", "percentile") if d in clim_ds.dims), None)
if qdim is not None:
    var = CLIM_VAR if CLIM_VAR in clim_ds else next(
        v for v in clim_ds.data_vars if qdim in clim_ds[v].dims)
    levels = np.asarray(clim_ds[qdim].values, dtype=float)
    scale = 100.0 if levels.max() <= 1.0 else 1.0           # 0.95 o 95
    clim = xr.Dataset({level_name(lv, scale): clim_ds[var].sel({qdim: lv}, drop=True)
                       for lv in levels})
    clim_units = clim_ds[var].attrs.get("units")
# Formato B: una variable por percentil
else:
    clim = clim_ds[list(CLIM_VAR_NAMES.values())].rename({v: k for k, v in CLIM_VAR_NAMES.items()})
    clim_units = clim_ds[next(iter(CLIM_VAR_NAMES.values()))].attrs.get("units")

print("percentiles disponibles:", list(clim.data_vars), "| variable:", var if qdim else "formato B")
missing_q = [q for q in REQUIRED_Q if q not in clim]
assert not missing_q, (f"faltan {missing_q} en {CLIM_FILE.name}; "
                       "recalcula los percentiles de 6 h incluyendo esos niveles")

clim_lat, clim_lon = infer_mswep_lat_lon_names(clim)
clim = clim.rename({clim_lat: LAT_NAME, clim_lon: LON_NAME})
clim = subset_mswep_domain(clim, **bbox)
clim = clim.reindex_like(P0, method="nearest", tolerance=1e-3)  # misma malla de 0.1°, sin interpolar

Q = {k: np.asarray(clim[k].values, dtype=float) for k in REQUIRED_Q}
assert all(v.shape == LON2D.shape for v in Q.values()), "climatología y MSWEP deben compartir malla"

finite = np.all([np.isfinite(v) for v in Q.values()], axis=0)
monotone = np.all([Q[a] <= Q[b] for a, b in zip(REQUIRED_Q[:-1], REQUIRED_Q[1:])], axis=0)
print(f"celdas con todos los percentiles válidos: {finite.mean():.1%}")
print(f"orden {' <= '.join(REQUIRED_Q)} en celdas válidas: {monotone[finite].mean():.2%}")
print({k: round(float(np.nanmedian(v)), 2) for k, v in Q.items()},
      "| unidades clim:", clim_units, "| unidades P6:", P6.attrs.get("units"))

<xarray.Dataset> Size: 87MB
Dimensions:           (quantile: 5, lat: 300, lon: 800, threshold: 4, hour: 4)
Coordinates:
  * quantile          (quantile) float64 40B 0.75 0.9 0.95 0.99 0.999
  * lat               (lat) float64 2kB 5.05 5.15 5.25 ... 34.75 34.85 34.95
  * lon               (lon) float64 6kB -139.9 -139.8 -139.8 ... -60.15 -60.05
  * threshold         (threshold) float64 32B 10.0 20.0 35.0 60.0
  * hour              (hour) int64 32B 0 6 12 18
Data variables:
    p_wet             (quantile, lat, lon) float32 5MB ...
    p_wet_raw         (quantile, lat, lon) float32 5MB ...
    n_wet             (lat, lon) int32 960kB ...
    frac_above_p_wet  (lat, lon, quantile) float32 5MB ...
    p_all             (quantile, lat, lon) float32 5MB ...
    n_valid           (lat, lon) int32 960kB ...
    wet_freq          (lat, lon) float32 960kB ...
    freq_above_abs    (threshold, lat, lon) float32 4MB ...
    p_wet_hour        (hour, quantile, lat, lon) float32 19MB ...
    p_wet_ra

In [ ]:
q_vmax = float(np.nanpercentile(Q[REQUIRED_Q[-1]], 98))
fig, axes = new_map_figure(1, len(REQUIRED_Q), wspace_in=0.5)
for ax, k in zip(axes[0], REQUIRED_Q):
    mesh = ax.pcolormesh(LON2D, LAT2D, Q[k], cmap=CMAP_PRECIP, vmin=0, vmax=q_vmax,
                         shading="auto", transform=PC, zorder=1)
    draw_track(ax)
    ax.set_title(k, fontsize=11)
hide_inner_labels(axes)
add_small_colorbar(axes[0, -1], mesh, "mm/6 h", extend="max")
finish_map_figure(fig, f"Percentiles de 6 h ({CLIM_VAR}, MSWEP V3.16) — "
                       "sin color: percentil no válido")
fig.savefig(FIG_DIR / "percentiles_6h.png", dpi=150, bbox_inches="tight")

## 6. Rampas de peligro de precipitación

`ramp_hazard` es una interpolación lineal por tramos entre nodos $(Q_k, h_k)$: $H_P=0$ para $P<Q_1$, $H_P=h_K$ para $P\geq Q_K$. Las celdas sin percentiles válidos reciben $H_P=0$ y se contabilizan.

Para pasar $H_P$ por el pipeline sin modificar el paquete se usa que, con $(Q_{90},Q_{95},Q_{99})=(0,\,0.5,\,1)$, la rampa de `itchi.precipitation` es la identidad en $[0,1]$: `compute_itchi_snapshot_from_grid(precipitation=H_P, q90=0, q95=0.5, q99=1)` usa exactamente $H_P$, y conserva la resolución de radios y el control de calidad del pipeline. Ambas propiedades se verifican abajo: identidad, y que `R3` reproduce la función del paquete con los percentiles reales.

In [ ]:
def ramp_hazard(p: np.ndarray, qs: dict, nodes: dict) -> tuple[np.ndarray, np.ndarray]:
    keys = list(nodes)                                       # nodos en orden creciente
    qk = [qs[k] for k in keys]
    hk = [float(nodes[k]) for k in keys]
    h = np.zeros_like(p, dtype=float)
    for (q_lo, h_lo), (q_hi, h_hi) in zip(zip(qk[:-1], hk[:-1]), zip(qk[1:], hk[1:])):
        seg = (p >= q_lo) & (p < q_hi)
        frac = (p - q_lo) / np.maximum(q_hi - q_lo, DEFAULT_EPSILON)
        h = np.where(seg, h_lo + (h_hi - h_lo) * frac, h)
    h = np.where(p >= qk[-1], hk[-1], h)
    valid = np.isfinite(p) & np.all([np.isfinite(q) for q in qk], axis=0)
    return np.clip(np.where(valid, h, 0.0), 0.0, 1.0), valid


# Verificación 1: identidad de la rampa del paquete con (0, 0.5, 1)
x = np.linspace(0.0, 1.0, 1001)
assert np.allclose(compute_precipitation_hazard(x, 0.0, 0.5, 1.0), x, atol=1e-9)

# Verificación 2: R3 coincide con compute_precipitation_hazard sobre P6 y percentiles reales
h3, valid = ramp_hazard(P0.values.astype(float), Q, RAMPS["R3"])
h_pkg = compute_precipitation_hazard(P0.values.astype(float), Q["Q90"], Q["Q95"], Q["Q99"])
assert np.allclose(h3[valid], h_pkg[valid], atol=1e-6)
print("verificaciones de la rampa: OK")

# Rampas en el espacio de percentiles de una celda típica (medianas del dominio)
q_med = {k: float(np.nanmedian(v)) for k, v in Q.items()}
p_axis = np.linspace(0, 1.3 * q_med[REQUIRED_Q[-1]], 600)
fig, ax = plt.subplots(figsize=(8, 4.2))
for name, nodes in RAMPS.items():
    color = RAMP_COLORS.get(name, None)
    ax.plot(p_axis, ramp_hazard(p_axis, q_med, nodes)[0], color=color, lw=2.2,
            ls=RAMP_LS.get(name, "-"), label=name)
    ax.plot([q_med[k] for k in nodes], list(nodes.values()), "o", color=color, ms=5, zorder=3)
for k, v in q_med.items():
    ax.axvline(v, color="0.65", ls=":", lw=0.9, zorder=0)
    ax.text(v, 1.01, k, transform=ax.get_xaxis_transform(), ha="center", va="bottom",
            fontsize=8.5, color="0.3")
ax.set(xlabel="P (mm/6 h)", ylabel="$H_P$", xlim=(0, p_axis[-1]), ylim=(0, 1.05))
ax.set_title("Rampas con los percentiles medianos del dominio", pad=16)
ax.grid(alpha=0.3, lw=0.6)
ax.legend(loc="lower right", frameon=False)
fig.savefig(FIG_DIR / "ramps.png", dpi=150, bbox_inches="tight")

## 7. ITCHI por snapshot de 6 h: 2 radios × 2 rampas

El viento (W06) y $H_P$ se calculan una vez por snapshot y se reutilizan en los cuatro experimentos. Productos de evento: `ITCHI_max` (máximo temporal) e `ITCHI_acc` (unión acotada $1-\prod_t(1-\mathrm{ITCHI}_t)$). `ITCHI_acc` a 6 h no es comparable con el `ITCHI_acc` diario del notebook 02: con cuatro veces más snapshots, la unión acotada se satura antes.

In [ ]:
WIND = {}
for _, row in track.iterrows():
    t = row["time"]
    radius = build_geometry_fields(LON2D, LAT2D, row["lon"], row["lat"])["radius_km"]
    WIND[t] = compute_w06_wind_hazard(radius, row["vmax_kt"], row["lat"], rmw_km=rmw_km(row),
                                      x1_km=float(row[f"{PREFIX}x1_km"]))
HP = {(t, r): ramp_hazard(PRECIP6[t], Q, nodes)[0] for t in PRECIP6 for r, nodes in RAMPS.items()}

FIELDS_F = ("ITCHI", "H_P", "H_dir", "H_ind")
FIELDS_B = ("M_direct", "M_indirect")


def run_snapshot(row: pd.Series, attribution: str, ramp: str):
    t = row["time"]
    radii = RADII[(t, attribution)]
    if radii is None:
        return None
    return compute_itchi_snapshot_from_grid(
        precipitation=HP[(t, ramp)], q90=0.0, q95=0.5, q99=1.0,      # identidad: H_P ya calculado
        lon=LON2D, lat=LAT2D, center_lon=float(row["lon"]), center_lat=float(row["lat"]),
        r34_by_quadrant={q: row[DEFAULT_R34_COLUMN_MAP[q]] for q in QUADRANTS}, r34_unit="nm",
        rocloud_by_quadrant=radii, rocloud_unit="km",
        wind_hazard_normalized=WIND[t], vmax_kt=float(row["vmax_kt"]), rmw_km=rmw_km(row),
        fallback_direct_radius_km=parametric_direct_fallback_km(row, f"{PREFIX}r34_km"),
        run_quality_control=True,
    )


def run_experiment(attribution: str, ramp: str):
    stack = {k: [] for k in FIELDS_F + FIELDS_B}
    times, rows = [], []
    for _, row in track.iterrows():
        res = run_snapshot(row, attribution, ramp)
        if res is None:
            continue
        region = np.asarray(res["M_direct"]) | np.asarray(res["M_indirect"])
        itchi = np.asarray(res["ITCHI"], dtype=float)
        _, valid = ramp_hazard(PRECIP6[row["time"]], Q, RAMPS[ramp])
        rows.append(dict(
            time=row["time"], attribution=attribution, ramp=ramp,
            itchi_max=float(itchi.max()),
            area_region_km2=float(CELL_AREA_KM2[region].sum()),
            area_itchi_gt0_km2=float(CELL_AREA_KM2[itchi > 0].sum()),
            area_itchi_gt05_km2=float(CELL_AREA_KM2[itchi > 0.5].sum()),
            mean_HP_region=float(np.asarray(res["H_P"])[region].mean()) if region.any() else np.nan,
            frac_region_sin_clim=float((~valid[region]).mean()) if region.any() else np.nan,
        ))
        for k in FIELDS_F:
            stack[k].append(np.asarray(res[k], dtype=np.float32))
        for k in FIELDS_B:
            stack[k].append(np.asarray(res[k], dtype=bool))
        times.append(row["time"])

    coords = {"time": times, LAT_NAME: P0[LAT_NAME].values, LON_NAME: P0[LON_NAME].values}
    dims = ("time", LAT_NAME, LON_NAME)
    ds = xr.Dataset({k: (dims, np.stack(v)) for k, v in stack.items()}, coords=coords)
    ds["ITCHI_max"] = compute_event_max(ds["ITCHI"], dim="time")
    ds["ITCHI_acc"] = compute_event_accumulated(ds["ITCHI"], dim="time")
    ds.attrs.update(case=CASE, sid=SID, atcf=C["atcf"], attribution=attribution, ramp=ramp,
                    ramp_nodes=str(RAMPS[ramp]), w06_config=W06_CHOICE, wind_profile="w06",
                    precipitation="MSWEP V3.16, 6-h accumulation (T-6h, T]",
                    climatology=CLIM_FILE.name)
    return ds, pd.DataFrame(rows)


events, snap_metrics = {}, []
for exp in EXPERIMENTS:
    events[exp], m = run_experiment(*exp)
    snap_metrics.append(m)
    print(exp, f"{events[exp].sizes['time']} snapshots",
          f"| ITCHI_max evento = {float(events[exp]['ITCHI_max'].max()):.3f}")
snap_metrics = pd.concat(snap_metrics, ignore_index=True)
snap_metrics.to_csv(DIAG_DIR / f"itchi_snapshot_metrics_{CASE.lower()}_{TAG}.csv", index=False)

('w06', 'R3') 30 snapshots | ITCHI_max evento = 1.000
('w06', 'R4') 30 snapshots | ITCHI_max evento = 1.000
('rocloud_w06floor', 'R3') 30 snapshots | ITCHI_max evento = 1.000
('rocloud_w06floor', 'R4') 30 snapshots | ITCHI_max evento = 1.000


## 8. Qué parte de la precipitación activa cada rampa

Fracción del área de la región de atribución `w06` (común a ambas hipótesis) en cada banda de percentiles, por snapshot. La banda $[Q_{75}, Q_{90})$ es la que solo activa `R4`.

In [ ]:
BANDS = ["<Q75", "Q75–Q90", "Q90–Q95", "Q95–Q99", "≥Q99"]
edges = [Q["Q75"], Q["Q90"], Q["Q95"], Q["Q99"]]
ref_ds = events[REFERENCE]
band_rows = []
for i, t in enumerate(ref_ds["time"].values):
    p = PRECIP6[pd.Timestamp(t)]
    region = (ref_ds["M_direct"][i] | ref_ds["M_indirect"][i]).values & finite
    idx = np.sum([p >= e for e in edges], axis=0)           # 0..4 = banda
    w = CELL_AREA_KM2[region]
    band_rows.append({b: float(w[idx[region] == k].sum() / w.sum()) for k, b in enumerate(BANDS)})
bands = pd.DataFrame(band_rows, index=pd.to_datetime(ref_ds["time"].values))

band_colors = ["0.90"] + [CMAP_PRECIP(x) for x in (0.30, 0.50, 0.72, 0.95)]
fig, ax = plt.subplots(figsize=(11, 4.2))
polys = ax.stackplot(bands.index, bands.T.values, labels=BANDS, colors=band_colors, lw=0)
ax.set(ylabel="fracción del área de atribución", ylim=(0, 1),
       xlim=(bands.index.min(), bands.index.max()))
ax.set_title(f"{C['name']} {C['season']}: bandas de P6 respecto a los percentiles de 6 h "
             f"(región {exp_label(*REFERENCE)})", fontsize=11)
format_time_axis(ax)
ax.legend(polys[::-1], BANDS[::-1], loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9,
          frameon=False, title="banda", title_fontsize=9)       # mismo orden que el apilado
fig.savefig(FIG_DIR / "precip_bands_in_region.png", dpi=150, bbox_inches="tight")
bands.mean().round(3)

## 9. Mapas de ITCHI_max por experimento

In [ ]:
SHOW_SIAT = True                            # también la versión en niveles SIAT

fig = plot_event_grid("ITCHI_max")
fig.savefig(FIG_DIR / "itchi_max_2x2.png", dpi=150, bbox_inches="tight")

if SHOW_SIAT:
    fig = plot_event_grid("ITCHI_max", siat=True)
    fig.savefig(FIG_DIR / "itchi_max_2x2_siat.png", dpi=150, bbox_inches="tight")

## 10. Efecto de la rampa y efecto del radio

Efecto de la rampa: `R4 − R3` con el mismo radio. Efecto del radio: `rocloud_w06floor − w06` con la misma rampa. Por construcción (sección 0) ambas diferencias deben ser ≥ 0; un mínimo negativo más allá del redondeo indica un error en los radios o en los nodos.

In [ ]:
r_old, r_new = list(RAMPS)
a_ref, a_alt = ATTRIBUTIONS
diffs = {
    f"{r_new} − {r_old} | {a}": events[(a, r_new)]["ITCHI_max"] - events[(a, r_old)]["ITCHI_max"]
    for a in ATTRIBUTIONS
}
diffs.update({
    f"{a_alt} − {a_ref} | {r}": events[(a_alt, r)]["ITCHI_max"] - events[(a_ref, r)]["ITCHI_max"]
    for r in RAMPS
})
for name, d in diffs.items():
    print(f"{name:35s} min = {float(d.min()):+.2e}  max = {float(d.max()):.3f}")

# Fila 1: efecto de la rampa; fila 2: efecto del radio. Escala común por fila.
names = list(diffs)
rows = [names[: len(ATTRIBUTIONS)], names[len(ATTRIBUTIONS):]]
fig, axes = new_map_figure(len(rows), max(len(r) for r in rows))
for i, row_names in enumerate(rows):
    vmax = max(float(np.nanmax(np.abs(diffs[n].values))) for n in row_names)
    for ax, n in zip(axes[i], row_names):
        draw_delta(ax, diffs[n].values, vmax)
        draw_track(ax)
        ax.set_title(n, fontsize=10)
hide_inner_labels(axes)
finish_map_figure(fig, f"{C['name']} {C['season']} — ΔITCHI$_{{max}}$ "
                       "(fila 1: efecto de la rampa; fila 2: efecto del radio)",
                  handles=map_legend_handles(regions=False, position=False))
fig.savefig(FIG_DIR / "itchi_max_diffs.png", dpi=150, bbox_inches="tight")

In [ ]:
# Snapshot de verificación: experimentos (filas) x campos (columnas), con las regiones
# exactas del pipeline (negro: directa; magenta: atribución) y P6 = Q95 en rojo
t = T_CHECK if T_CHECK in PRECIP6 else min(PRECIP6, key=lambda s: abs(s - T_CHECK))
fig = plot_snapshot_grid(t, SNAP_FIELDS)
if fig is not None:
    fig.savefig(FIG_DIR / f"snapshot_{t:%Y%m%d%H}.png", dpi=150, bbox_inches="tight")

### Mapas individuales (a detalle, sin subplots)

`plot_snapshot_map(t, exp, campo)` y `plot_event_map(exp, campo)` dibujan un solo mapa con el mismo estilo. Campos de snapshot: `P6`, `H_P`, `H_dir`, `H_ind`, `ITCHI`, `SIAT`. Devuelven `None` (con aviso) si la posición o el experimento no existen, sin detener la ejecución. `compare_attribution=True` añade en línea discontinua la región de atribución de la otra hipótesis con la misma rampa.

In [ ]:
SAVE_SINGLE_MAPS = False                    # True: un PNG por experimento y campo

fig = plot_snapshot_map(t, ("rocloud_w06floor", "R4"), "ITCHI", compare_attribution=True)
fig = plot_event_map(("rocloud_w06floor", "R4"), "ITCHI_max")

if SAVE_SINGLE_MAPS:
    out = FIG_DIR / f"single_{t:%Y%m%d%H}"
    out.mkdir(exist_ok=True)
    skipped = []
    for exp in EXPERIMENTS:
        for field in ("P6", "H_P", "H_dir", "H_ind", "ITCHI", "SIAT"):
            fig = plot_snapshot_map(t, exp, field)
            if fig is None:
                skipped.append((exp, field))
                continue
            fig.savefig(out / f"{exp[0]}_{exp[1]}_{field}.png", dpi=150, bbox_inches="tight")
            plt.close(fig)
    print("omitidos:", skipped)

## 11. Series temporales y resumen del caso

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)
fig.subplots_adjust(hspace=0.12, top=0.90)
for a, r in EXPERIMENTS:                      # mismo orden de leyenda que los mapas
    d = snap_metrics[(snap_metrics["attribution"] == a) & (snap_metrics["ramp"] == r)]
    style = exp_style(a, r)
    axes[0].plot(d["time"], d["itchi_max"], lw=1.6, **style)
    axes[1].plot(d["time"], d["area_itchi_gt05_km2"] / 1e3, lw=1.6, **style)
    axes[2].plot(d["time"], d["mean_HP_region"], lw=1.6, **style)
axes[0].set_ylabel("ITCHI máx.")
axes[1].set_ylabel("área ITCHI>0.5\n(10³ km²)")
axes[2].set_ylabel("$\\overline{H_P}$ en la región")
for ax in axes:
    format_time_axis(ax)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels), fontsize=9, frameon=False,
           bbox_to_anchor=(0.5, 0.955))
fig.suptitle(f"{C['name']} {C['season']}: métricas por snapshot (color: radio; línea: rampa)",
             y=0.99)
fig.savefig(FIG_DIR / "snapshot_timeseries.png", dpi=150, bbox_inches="tight")

In [ ]:
def event_summary(ev: xr.Dataset, ref: xr.Dataset) -> dict:
    imax = ev["ITCHI_max"].values
    diff = imax - ref["ITCHI_max"].values
    return {
        "snapshots": int(ev.sizes["time"]),
        "area_ITCHI_max>0_km2": float(CELL_AREA_KM2[imax > 0].sum()),
        "area_ITCHI_max>0.25_km2": float(CELL_AREA_KM2[imax > 0.25].sum()),
        "area_ITCHI_max>0.5_km2": float(CELL_AREA_KM2[imax > 0.5].sum()),
        "media_ITCHI_max(>0)": float(imax[imax > 0].mean()) if (imax > 0).any() else np.nan,
        "area_|Δ|>0.1_vs_ref_km2": float(CELL_AREA_KM2[np.abs(diff) > 0.1].sum()),
        "ΔITCHI_max_medio(ref>0 o exp>0)": float(diff[(imax > 0) | (ref["ITCHI_max"].values > 0)].mean()),
    }


summary = pd.DataFrame({f"{a} · {r}": event_summary(events[(a, r)], events[REFERENCE])
                        for a, r in EXPERIMENTS}).T
summary.insert(0, "case", CASE)
summary.attrs["reference"] = f"{REFERENCE[0]} · {REFERENCE[1]}"
summary.to_csv(DIAG_DIR / f"itchi_summary_{CASE.lower()}_{TAG}.csv")
display(summary.round(3))

for (a, r), ev in events.items():
    ev.to_netcdf(OUT_DIR / f"itchi_{CASE.lower()}_{a}_{r}_{TAG}.nc",
                 encoding={k: {"zlib": True, "complevel": 4} for k in ev.data_vars})

,case,snapshots,area_ITCHI_max>0_km2,area_ITCHI_max>0.25_km2,area_ITCHI_max>0.5_km2,media_ITCHI_max(>0),area_|Δ|>0.1_vs_ref_km2,ΔITCHI_max_medio(ref>0 o exp>0)
w06 · R3,CRISTOBAL2020,30.0,2368475.888,2141426.345,1932810.015,0.764,0.000,0.000
w06 · R4,CRISTOBAL2020,30.0,2989844.691,2355391.715,1936449.227,0.651,603564.189,0.042
rocloud_w06floor · R3,CRISTOBAL2020,30.0,2544098.609,2277142.045,2039959.157,0.752,184821.288,0.038
rocloud_w06floor · R4,CRISTOBAL2020,30.0,3284128.662,2531014.436,2043598.369,0.631,860524.948,0.074


In [ ]:
# Tras correr ambos casos: tabla conjunta
paths = sorted(DIAG_DIR.glob(f"itchi_summary_*_{TAG}.csv"))
pd.concat([pd.read_csv(p, index_col=0) for p in paths]).round(3) if paths else None

,case,snapshots,area_ITCHI_max>0_km2,area_ITCHI_max>0.25_km2,area_ITCHI_max>0.5_km2,media_ITCHI_max(>0),area_|Δ|>0.1_vs_ref_km2,ΔITCHI_max_medio(ref>0 o exp>0)
w06 · R3,CRISTOBAL2020,30.0,2368475.888,2141426.345,1932810.015,0.764,0.000,0.000
w06 · R4,CRISTOBAL2020,30.0,2989844.691,2355391.715,1936449.227,0.651,603564.189,0.042
rocloud_w06floor · R3,CRISTOBAL2020,30.0,2544098.609,2277142.045,2039959.157,0.752,184821.288,0.038
rocloud_w06floor · R4,CRISTOBAL2020,30.0,3284128.662,2531014.436,2043598.369,0.631,860524.948,0.074
